# 🏏 EE604 Image Processing Course Project: Cricket Shot Recognition (`CricIP-FusionNet`)
**100% Self-Contained Google Colab Notebook for Training, Ablation Studies, Paper Figures & Live Gradio Demo**

### Quick Setup Instructions:
1. In Google Colab, make sure **T4 GPU** is selected (bottom right shows `T4`).
2. Click **Runtime $\rightarrow$ Run all**! Every module (`src/ip_pipeline.py`, `src/dataset.py`, `src/model.py`, `train_eval.py`, `app.py`) is automatically created inside Colab by the cells below.

In [ ]:
# Step 1: Mount Google Drive & Automatically Locate 'CricShot10 dataset'
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CANDIDATE_PATHS = [
    '/content/drive/MyDrive/CricShot10 dataset',
    '/content/drive/MyDrive/Soumyajit Halder/CricShot10 dataset',
    '/content/drive/.shortcut-targets-by-id/1DPHURwQk5R8blgjM8VNz6Q68LqckxljX/CricShot10 dataset',
    '/content/drive/.shortcut-targets-by-id/1DPHURwQk5R8blgjM8VNz6Q68LqckxljX',
    '/content/drive/MyDrive/CricShot10',
    './CricShot10 dataset',
]

DATASET_ROOT = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATASET_ROOT = p
        break

if DATASET_ROOT is None and os.path.exists('/content/drive/MyDrive'):
    for root, dirs, _ in os.walk('/content/drive/MyDrive', followlinks=True):
        lower_dirs = [d.lower() for d in dirs]
        if 'cover' in lower_dirs and 'sweep' in lower_dirs:
            DATASET_ROOT = root
            break

os.makedirs('src', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
with open('src/__init__.py', 'w') as f:
    f.write('')

print('Detected DATASET_ROOT:', DATASET_ROOT)

In [ ]:
# Step 2: Install Dependencies
!pip install -q gradio scikit-image seaborn tqdm

In [ ]:
%%writefile src/ip_pipeline.py
from typing import Dict, List, Tuple
import cv2
import numpy as np

CRICSHOT10_CLASSES: List[str] = [
    'cover', 'defense', 'flick', 'hook', 'late_cut',
    'lofted', 'pull', 'square_cut', 'straight', 'sweep'
]

class CricketIPPipeline:
    def __init__(
        self,
        frame_size: Tuple[int, int] = (224, 224),
        num_keyframes: int = 8,
        clahe_clip_limit: float = 2.5,
        clahe_grid_size: Tuple[int, int] = (8, 8),
        mhi_tau: float = 8.0,
        mhi_diff_thresh: int = 22,
        hoof_bins: int = 36,
        hough_bins: int = 18,
    ) -> None:
        self.frame_size = frame_size
        self.num_keyframes = num_keyframes
        self.clahe_clip_limit = clahe_clip_limit
        self.clahe_grid_size = clahe_grid_size
        self.mhi_tau = float(mhi_tau)
        self.mhi_diff_thresh = mhi_diff_thresh
        self.hoof_bins = hoof_bins
        self.hough_bins = hough_bins
        self.hog = cv2.HOGDescriptor(
            _winSize=(64, 64), _blockSize=(16, 16),
            _blockStride=(16, 16), _cellSize=(8, 8), _nbins=9
        )

    def sample_keyframes(self, video_path: str) -> List[np.ndarray]:
        cap = cv2.VideoCapture(video_path)
        raw_frames = []
        while True:
            ret, frame_bgr = cap.read()
            if not ret or frame_bgr is None:
                break
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frame_rgb = cv2.resize(frame_rgb, self.frame_size, interpolation=cv2.INTER_AREA)
            raw_frames.append(frame_rgb)
        cap.release()
        if not raw_frames:
            raise ValueError(f'No readable frames in {video_path}')
        total = len(raw_frames)
        active = raw_frames[int(0.1*total):int(0.9*total)] if total >= 16 else raw_frames
        idxs = np.linspace(0, len(active) - 1, self.num_keyframes, dtype=int)
        return [active[i] for i in idxs]

    def apply_clahe_lab(self, rgb_frame: np.ndarray) -> np.ndarray:
        lab = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=self.clahe_clip_limit, tileGridSize=self.clahe_grid_size)
        l_eq = clahe.apply(l)
        return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2RGB)

    def suppress_crowd_and_segment_roi(self, rgb_frame: np.ndarray):
        h, w, _ = rgb_frame.shape
        hsv = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2HSV)
        green_mask = cv2.inRange(hsv, np.array([28, 30, 35]), np.array([95, 255, 255]))
        pitch_mask = cv2.inRange(hsv, np.array([10, 20, 80]), np.array([28, 170, 255]))
        comb = cv2.bitwise_or(green_mask, pitch_mask)
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        morphed = cv2.morphologyEx(comb, cv2.MORPH_CLOSE, k_close)
        yy, xx = np.meshgrid(np.linspace(-1, 1, h), np.linspace(-1, 1, w), indexing='ij')
        prior = np.exp(-((xx / 0.65) ** 2 + ((yy - 0.05) / 0.75) ** 2))
        soft = np.clip(0.45 * (morphed / 255.0) + 0.75 * prior, 0.20, 1.0).astype(np.float32)
        soft = cv2.GaussianBlur(soft, (11, 11), 0)
        suppressed = np.clip(rgb_frame.astype(np.float32) * soft[..., None], 0, 255).astype(np.uint8)
        chsv = hsv[h//4:3*h//4, w//4:3*w//4]
        stats = np.array([
            float(np.mean(green_mask > 0)), float(np.mean(pitch_mask > 0)),
            float(np.mean(chsv[:, :, 0]) / 180.0), float(np.mean(chsv[:, :, 1]) / 255.0),
            float(np.mean(chsv[:, :, 2]) / 255.0), float(np.std(chsv[:, :, 2]) / 255.0)
        ], dtype=np.float32)
        return suppressed, (soft * 255).astype(np.uint8), stats

    def compute_mhi_and_mei(self, frames: List[np.ndarray]):
        h, w, _ = frames[0].shape
        mhi = np.zeros((h, w), dtype=np.float32)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        grays = [cv2.GaussianBlur(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), (5, 5), 0) for f in frames]
        for t in range(1, len(grays)):
            diff = cv2.absdiff(grays[t], grays[t - 1])
            _, mask = cv2.threshold(diff, self.mhi_diff_thresh, 1, cv2.THRESH_BINARY)
            mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, kernel)
            mhi = np.where(mask > 0, self.mhi_tau, np.maximum(0.0, mhi - 1.0))
        mhi_norm = np.clip((mhi / self.mhi_tau) * 255.0, 0, 255).astype(np.uint8)
        mei_mask = np.where(mhi_norm > 0, 255, 0).astype(np.uint8)
        mhi_col = cv2.applyColorMap(mhi_norm, cv2.COLORMAP_TURBO)
        mhi_col[mhi_norm == 0] = 0
        return mhi_norm, mei_mask, cv2.cvtColor(mhi_col, cv2.COLOR_BGR2RGB)

    def compute_optical_flow_and_hoof(self, frames: List[np.ndarray]):
        h, w, _ = frames[0].shape
        grays = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in frames]
        r0, r1, c0, c1 = int(0.15*h), int(0.85*h), int(0.20*w), int(0.80*w)
        hoof_hist = np.zeros(self.hoof_bins, dtype=np.float32)
        accum_flow = np.zeros((h, w, 2), dtype=np.float32)
        peak_mag, peak_flow, mags = -1.0, np.zeros((h, w, 2), dtype=np.float32), []
        for t in range(1, len(grays)):
            flow = cv2.calcOpticalFlowFarneback(grays[t-1], grays[t], None, 0.5, 3, 15, 3, 5, 1.2, 0)
            flow[:, :, 0] -= np.median(flow[:15, :, 0])
            flow[:, :, 1] -= np.median(flow[:15, :, 1])
            accum_flow += flow
            roi = flow[r0:r1, c0:c1]
            mag, ang = cv2.cartToPolar(roi[:, :, 0], roi[:, :, 1], angleInDegrees=False)
            act = mag > 1.0
            if np.any(act):
                hist, _ = np.histogram(ang[act], bins=self.hoof_bins, range=(0, 2*np.pi), weights=mag[act])
                hoof_hist += hist.astype(np.float32)
            s = float(np.sum(mag))
            mags.append(s / float(mag.size))
            if s > peak_mag:
                peak_mag, peak_flow = s, flow.copy()
        hoof_norm = hoof_hist / (np.linalg.norm(hoof_hist) + 1e-6)
        avg_f = accum_flow / max(1, len(grays) - 1)
        vh = float(np.mean(np.abs(avg_f[r0:r1, c0:c1, 1])) / (np.mean(np.abs(avg_f[r0:r1, c0:c1, 0])) + 1e-5))
        kstats = np.array([
            np.tanh(float(np.mean(mags))/5.0), np.tanh(float(np.max(mags))/10.0),
            float(np.argmax(mags))/max(1, len(mags)-1), np.tanh(vh)
        ], dtype=np.float32)
        mv, av = cv2.cartToPolar(peak_flow[:, :, 0], peak_flow[:, :, 1], angleInDegrees=True)
        hsv = np.zeros((h, w, 3), dtype=np.uint8)
        hsv[:, :, 0] = (av / 2.0).astype(np.uint8)
        hsv[:, :, 1] = 255
        hsv[:, :, 2] = cv2.normalize(mv, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return np.concatenate([hoof_norm, kstats]), cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB), avg_f

    def extract_canny_hough_and_hog(self, peak_rgb: np.ndarray, mhi_norm: np.ndarray):
        h, w, _ = peak_rgb.shape
        gray = cv2.GaussianBlur(cv2.cvtColor(peak_rgb, cv2.COLOR_RGB2GRAY), (5, 5), 1.0)
        med = float(np.median(gray))
        edges = cv2.Canny(gray, int(max(25, 0.66*med)), int(min(220, 1.33*med)))
        roi_mask = np.zeros_like(edges)
        roi_mask[int(0.15*h):int(0.88*h), int(0.22*w):int(0.78*w)] = 255
        roi_edges = cv2.bitwise_and(edges, roi_mask)
        lines = cv2.HoughLinesP(roi_edges, 1.0, np.pi/180.0, 22, minLineLength=16, maxLineGap=8)
        h_hist = np.zeros(self.hough_bins, dtype=np.float32)
        overlay = (peak_rgb.astype(np.float32) * 0.45).astype(np.uint8)
        overlay[roi_edges > 0] = [0, 210, 255]
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                length = float(np.hypot(x2 - x1, y2 - y1))
                if length < 12.0:
                    continue
                ang = (np.degrees(np.arctan2(y2 - y1, x2 - x1)) + 180.0) % 180.0
                b = min(self.hough_bins - 1, int((ang / 180.0) * self.hough_bins))
                h_hist[b] += length
                cv2.line(overlay, (x1, y1), (x2, y2), (255, 215, 0) if length > 28 else (50, 255, 120), 2, cv2.LINE_AA)
        h_norm = h_hist / (np.linalg.norm(h_hist) + 1e-6)
        p = h_hist / (np.sum(h_hist) + 1e-6)
        ent = float(-np.sum(p * np.log(p + 1e-8)) / np.log(self.hough_bins))
        dom = float(np.argmax(h_hist)) / float(self.hough_bins)
        comp = cv2.resize(cv2.addWeighted(mhi_norm, 0.7, roi_edges, 0.3, 0), (64, 64))
        hog_vec = self.hog.compute(comp).ravel().reshape(16, 36).mean(axis=0).astype(np.float32)
        hog_vec /= (np.linalg.norm(hog_vec) + 1e-6)
        return np.concatenate([h_norm, np.array([ent, dom], dtype=np.float32), hog_vec]), overlay, h_norm

    def process_video(self, video_path: str) -> Dict[str, np.ndarray]:
        raw = self.sample_keyframes(video_path)
        clahe = [self.apply_clahe_lab(f) for f in raw]
        supp, pstats = [], []
        mid = len(clahe) // 2
        mid_mask = None
        for i, cf in enumerate(clahe):
            s, m, ps = self.suppress_crowd_and_segment_roi(cf)
            supp.append(s)
            pstats.append(ps)
            if i == mid:
                mid_mask = m
        mhi_g, mei_m, mhi_c = self.compute_mhi_and_mei(supp)
        hoof_d, flow_rgb, _ = self.compute_optical_flow_and_hoof(supp)
        geom_d, h_over, h_hist = self.extract_canny_hough_and_hog(supp[mid], mhi_g)
        flow_g = cv2.cvtColor(flow_rgb, cv2.COLOR_RGB2GRAY)
        motion_comp = np.stack([mhi_g, flow_g, mei_m], axis=-1)
        ip_vec = np.concatenate([hoof_d, geom_d, np.mean(pstats, axis=0)]).astype(np.float32)
        return {
            'raw_keyframes': np.stack(raw), 'clahe_keyframes': np.stack(clahe),
            'suppressed_keyframes': np.stack(supp), 'roi_mask': mid_mask,
            'mhi_gray': mhi_g, 'mei_mask': mei_m, 'mhi_color': mhi_c,
            'flow_rgb': flow_rgb, 'hough_overlay': h_over,
            'motion_composite': motion_comp, 'hoof_hist': hoof_d[:self.hoof_bins],
            'hough_hist': h_hist, 'ip_feature_vector': ip_vec
        }


In [ ]:
%%writefile src/dataset.py
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from .ip_pipeline import CRICSHOT10_CLASSES, CricketIPPipeline

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalize_image_tensor(img_uint8: np.ndarray) -> torch.Tensor:
    arr = img_uint8.astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    if arr.ndim == 3:
        return torch.from_numpy(np.transpose(arr, (2, 0, 1)).copy())
    elif arr.ndim == 4:
        return torch.from_numpy(np.transpose(arr, (0, 3, 1, 2)).copy())
    raise ValueError(f'Unsupported shape: {img_uint8.shape}')

def discover_cricshot10_videos(dataset_root: str, max_per_class: Optional[int] = None) -> List[Tuple[str, int, str]]:
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f'Dataset directory not found: {dataset_root}')
    samples: List[Tuple[str, int, str]] = []
    valid_exts = {'.avi', '.mp4', '.mkv', '.mov'}
    for cls_idx, cls_name in enumerate(CRICSHOT10_CLASSES):
        cls_dir = root / cls_name
        if not cls_dir.exists():
            cands = [d for d in root.iterdir() if d.is_dir() and d.name.lower().replace(' ', '_') == cls_name.lower()]
            if cands:
                cls_dir = cands[0]
            else:
                continue
        cls_files = sorted([str(p) for p in cls_dir.rglob('*') if p.suffix.lower() in valid_exts])
        if max_per_class is not None:
            cls_files = cls_files[:max_per_class]
        for fpath in cls_files:
            samples.append((fpath, cls_idx, cls_name))
    return samples

class CricShot10Dataset(Dataset):
    def __init__(
        self,
        samples: List[Tuple[str, int, str]],
        pipeline: Optional[CricketIPPipeline] = None,
        cache_dir: Optional[str] = None,
        use_clahe_suppression: bool = True,
        augment: bool = False,
    ) -> None:
        self.samples = samples
        self.pipeline = pipeline or CricketIPPipeline()
        self.cache_dir = Path(cache_dir) if cache_dir else None
        self.use_clahe_suppression = use_clahe_suppression
        self.augment = augment
        if self.cache_dir is not None:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

    def __len__(self) -> int:
        return len(self.samples)

    def _get_cache_path(self, video_path: str, cls_name: str) -> Optional[Path]:
        if self.cache_dir is None:
            return None
        return self.cache_dir / f'{cls_name}_{Path(video_path).stem}.npz'

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        video_path, label, cls_name = self.samples[idx]
        cache_path = self._get_cache_path(video_path, cls_name)
        data_dict = None
        if cache_path is not None and cache_path.exists():
            try:
                loaded = np.load(str(cache_path))
                data_dict = {
                    'raw_keyframes': loaded['raw_keyframes'],
                    'suppressed_keyframes': loaded['suppressed_keyframes'],
                    'motion_composite': loaded['motion_composite'],
                    'ip_feature_vector': loaded['ip_feature_vector'],
                }
            except Exception:
                data_dict = None
        if data_dict is None:
            processed = self.pipeline.process_video(video_path)
            data_dict = {
                'raw_keyframes': processed['raw_keyframes'],
                'suppressed_keyframes': processed['suppressed_keyframes'],
                'motion_composite': processed['motion_composite'],
                'ip_feature_vector': processed['ip_feature_vector'],
            }
            if cache_path is not None:
                np.savez_compressed(
                    str(cache_path),
                    raw_keyframes=data_dict['raw_keyframes'],
                    suppressed_keyframes=data_dict['suppressed_keyframes'],
                    motion_composite=data_dict['motion_composite'],
                    ip_feature_vector=data_dict['ip_feature_vector'],
                )
        keyframes_np = data_dict['suppressed_keyframes'] if self.use_clahe_suppression else data_dict['raw_keyframes']
        motion_np = data_dict['motion_composite']
        ip_vec_np = data_dict['ip_feature_vector'].copy()
        if self.augment and np.random.rand() < 0.5:
            keyframes_np = keyframes_np[:, :, ::-1, :].copy()
            motion_np = motion_np[:, ::-1, :].copy()
        return {
            'keyframes': normalize_image_tensor(keyframes_np),
            'motion_map': normalize_image_tensor(motion_np),
            'ip_vec': torch.from_numpy(ip_vec_np).float(),
            'label': torch.tensor(label, dtype=torch.long),
        }

def generate_demo_cricket_videos(output_dir: str) -> List[str]:
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    stroke_profiles = {
        'cover': (75, 15, -35, -10, (-45, 25)),
        'defense': (88, 82, 0, 5, (0, 10)),
        'flick': (80, 135, 30, -5, (45, 20)),
        'hook': (45, 165, 38, -32, (50, -35)),
        'late_cut': (60, 10, -40, 12, (-50, -10)),
        'lofted': (85, 145, 15, -42, (10, -55)),
        'pull': (35, 155, 42, -12, (55, -10)),
        'square_cut': (40, 5, -45, 0, (-60, 0)),
        'straight': (88, 65, 2, -25, (0, 45)),
        'sweep': (25, 160, 45, 18, (50, 15)),
    }
    created = []
    h, w, num_frames = 224, 224, 16
    for cls_name, (ang0, ang1, sdx, sdy, (bdx, bdy)) in stroke_profiles.items():
        cls_folder = out_path / cls_name
        cls_folder.mkdir(parents=True, exist_ok=True)
        video_file = cls_folder / f'sample_{cls_name}.mp4'
        writer = cv2.VideoWriter(str(video_file), cv2.VideoWriter_fourcc(*'mp4v'), 12.0, (w, h))
        rng = np.random.RandomState(abs(hash(cls_name)) % 10000)
        crowd = rng.randint(40, 180, size=(55, w, 3), dtype=np.uint8)
        for t in range(num_frames):
            alpha = t / float(num_frames - 1)
            frame = np.zeros((h, w, 3), dtype=np.uint8)
            frame[:55, :] = crowd
            frame[55:, :] = (45, 135, 55)
            pts = np.array([[82, 65], [142, 65], [162, 220], [62, 220]], dtype=np.int32)
            cv2.fillPoly(frame, [pts], (135, 175, 200))
            cv2.line(frame, (72, 145), (152, 145), (245, 245, 245), 2)
            cx, cy = int(112 + alpha * sdx * 0.25), int(115 + alpha * sdy * 0.20)
            cv2.line(frame, (cx - 6, cy + 12), (cx - 10, cy + 45), (235, 235, 235), 6)
            cv2.line(frame, (cx + 6, cy + 12), (cx + 10, cy + 45), (235, 235, 235), 6)
            cv2.rectangle(frame, (cx - 10, cy - 16), (cx + 10, cy + 14), (190, 85, 35), -1)
            cv2.circle(frame, (cx, cy - 24), 9, (90, 40, 20), -1)
            cur_ang = np.radians(ang0 + alpha * (ang1 - ang0))
            wx, wy = int(cx + 8 + alpha * sdx * 0.35), int(cy - 2 + alpha * sdy * 0.35)
            tx, ty = int(wx + 38 * np.cos(cur_ang)), int(wy + 38 * np.sin(cur_ang))
            cv2.line(frame, (cx, cy - 8), (wx, wy), (220, 220, 220), 4)
            cv2.line(frame, (wx, wy), (tx, ty), (70, 165, 230), 6, cv2.LINE_AA)
            writer.write(frame)
        writer.release()
        created.append(str(video_file))
    return created


In [ ]:
%%writefile src/model.py
from typing import Dict, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class TemporalAttentionPool(nn.Module):
    def __init__(self, feat_dim: int) -> None:
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 4),
            nn.Tanh(),
            nn.Linear(feat_dim // 4, 1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        scores = self.attn(x).squeeze(-1)
        weights = F.softmax(scores, dim=-1)
        pooled = torch.bmm(weights.unsqueeze(1), x).squeeze(1)
        return pooled, weights

class ClassicalIPMLP(nn.Module):
    def __init__(self, ip_feat_dim: int = 102, embed_dim: int = 128, num_classes: int = 10, dropout: float = 0.3) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(ip_feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
        )
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, ip_vec: torch.Tensor) -> Dict[str, torch.Tensor]:
        feat = self.encoder(ip_vec)
        return {'logits': self.classifier(feat), 'features': feat}

class RGBBaselineNet(nn.Module):
    def __init__(self, num_classes: int = 10, pretrained: bool = True, use_temporal_attn: bool = False, dropout: float = 0.3) -> None:
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.use_temporal_attn = use_temporal_attn
        if use_temporal_attn:
            self.temp_pool = TemporalAttentionPool(self.feat_dim)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feat_dim, num_classes))

    def forward(self, keyframes: torch.Tensor) -> Dict[str, torch.Tensor]:
        b, t, c, h, w = keyframes.shape
        frame_feats = self.backbone(keyframes.view(b * t, c, h, w)).view(b, t, self.feat_dim)
        if self.use_temporal_attn:
            vid_feat, temp_weights = self.temp_pool(frame_feats)
        else:
            vid_feat = frame_feats.mean(dim=1)
            temp_weights = torch.full((b, t), 1.0 / t, device=keyframes.device)
        return {'logits': self.head(vid_feat), 'features': vid_feat, 'temporal_weights': temp_weights}

class GatedCrossModalAttention(nn.Module):
    def __init__(self, embed_dim: int = 256, num_modalities: int = 3) -> None:
        super().__init__()
        self.embed_dim = embed_dim
        self.gate_net = nn.Sequential(
            nn.Linear(embed_dim * num_modalities, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, num_modalities),
        )
        self.context_proj = nn.Sequential(
            nn.Linear(embed_dim * num_modalities, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )

    def forward(self, e_rgb: torch.Tensor, e_motion: torch.Tensor, e_geom: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        concat = torch.cat([e_rgb, e_motion, e_geom], dim=-1)
        alpha = F.softmax(self.gate_net(concat), dim=-1)
        stacked = torch.stack([e_rgb, e_motion, e_geom], dim=1)
        weighted_sum = torch.sum(alpha.unsqueeze(-1) * stacked, dim=1)
        fused = F.layer_norm(weighted_sum + 0.5 * self.context_proj(concat), (self.embed_dim,))
        return fused, alpha

class CricIPFusionNet(nn.Module):
    def __init__(self, ip_feat_dim: int = 102, embed_dim: int = 256, num_classes: int = 10, pretrained: bool = True, use_geom_stream: bool = True, dropout: float = 0.35) -> None:
        super().__init__()
        self.use_geom_stream = use_geom_stream
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        rgb_backbone = models.resnet18(weights=weights)
        rgb_dim = rgb_backbone.fc.in_features
        rgb_backbone.fc = nn.Identity()
        self.rgb_backbone = rgb_backbone
        self.temp_pool = TemporalAttentionPool(rgb_dim)
        self.rgb_proj = nn.Sequential(nn.Linear(rgb_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())

        motion_backbone = models.resnet18(weights=weights)
        motion_dim = motion_backbone.fc.in_features
        motion_backbone.fc = nn.Identity()
        self.motion_backbone = motion_backbone
        self.motion_proj = nn.Sequential(nn.Linear(motion_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())

        self.geom_encoder = nn.Sequential(
            nn.Linear(ip_feat_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.gcam = GatedCrossModalAttention(embed_dim=embed_dim, num_modalities=3)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(embed_dim, embed_dim // 2), nn.GELU(), nn.Linear(embed_dim // 2, num_classes)
        )

    def forward(self, keyframes: torch.Tensor, motion_map: torch.Tensor, ip_vec: torch.Tensor) -> Dict[str, torch.Tensor]:
        b, t, c, h, w = keyframes.shape
        rgb_feats = self.rgb_backbone(keyframes.view(b * t, c, h, w)).view(b, t, -1)
        rgb_pooled, temp_weights = self.temp_pool(rgb_feats)
        e_rgb = self.rgb_proj(rgb_pooled)
        e_motion = self.motion_proj(self.motion_backbone(motion_map))
        e_geom = self.geom_encoder(ip_vec) if self.use_geom_stream else torch.zeros_like(e_rgb)
        fused, modality_weights = self.gcam(e_rgb, e_motion, e_geom)
        logits = self.classifier(fused)
        return {'logits': logits, 'fused_features': fused, 'modality_weights': modality_weights, 'temporal_weights': temp_weights}


In [ ]:
%%writefile train_eval.py
import argparse
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.dataset import CricShot10Dataset, discover_cricshot10_videos, generate_demo_cricket_videos
from src.ip_pipeline import CRICSHOT10_CLASSES, CricketIPPipeline
from src.model import ClassicalIPMLP, CricIPFusionNet, RGBBaselineNet

def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_ip_pipeline_figure(video_path: str, class_name: str, pipeline: CricketIPPipeline, save_path: str) -> None:
    out = pipeline.process_video(video_path)
    mid = len(out['raw_keyframes']) // 2
    fig, axes = plt.subplots(2, 3, figsize=(14, 8.2), dpi=200)
    fig.suptitle(f"Proposed Multi-Stage Image Processing Pipeline — Class: '{class_name.upper()}'", fontsize=14, fontweight='bold', y=0.98)
    axes[0, 0].imshow(out['raw_keyframes'][mid]); axes[0, 0].set_title('(a) Raw Broadcast Keyframe', fontweight='bold'); axes[0, 0].axis('off')
    axes[0, 1].imshow(out['suppressed_keyframes'][mid]); axes[0, 1].set_title('(b) LAB-CLAHE & Crowd-Suppressed ROI', fontweight='bold'); axes[0, 1].axis('off')
    axes[0, 2].imshow(out['mhi_color']); axes[0, 2].set_title(r'(c) Temporal Motion History $H_\tau(x,y,t)$', fontweight='bold'); axes[0, 2].axis('off')
    axes[1, 0].imshow(out['flow_rgb']); axes[1, 0].set_title('(d) Dense Farneback Optical Flow (HSV)', fontweight='bold'); axes[1, 0].axis('off')
    axes[1, 1].imshow(out['hough_overlay']); axes[1, 1].set_title('(e) Canny Edges + Hough Bat/Limb Lines', fontweight='bold'); axes[1, 1].axis('off')
    ax6 = axes[1, 2]
    ax6.bar(np.linspace(0, 360, len(out['hoof_hist']), endpoint=False), out['hoof_hist'], width=8.0, alpha=0.75, color='#1f77b4', label='36-Bin HOOF')
    ax6.bar(np.linspace(0, 180, len(out['hough_hist']), endpoint=False)*2.0, out['hough_hist'], width=12.0, alpha=0.65, color='#ff7f0e', label='18-Bin Hough Bat Angle')
    ax6.set_title('(f) Scale-Invariant HOOF & Hough Descriptors', fontweight='bold')
    ax6.set_xlabel('Orientation Angle (Degrees)'); ax6.set_ylabel('Normalized Energy'); ax6.legend(loc='upper right', fontsize=8.5); ax6.grid(alpha=0.3)
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

def train_and_evaluate_variant(variant_name, model, train_loader, val_loader, device, epochs=12, lr=3e-4, mode='fusion'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    best_acc, best_f1 = -1.0, 0.0
    best_preds, best_targets, best_mod_weights, best_state = np.array([]), np.array([]), np.zeros((1, 3), dtype=np.float32), None
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            keyframes, motion_map, ip_vec, labels = batch['keyframes'].to(device), batch['motion_map'].to(device), batch['ip_vec'].to(device), batch['label'].to(device)
            optimizer.zero_grad()
            out = model(ip_vec) if mode == 'ip_only' else (model(keyframes) if mode == 'rgb_only' else model(keyframes, motion_map, ip_vec))
            loss = criterion(out['logits'], labels)
            loss.backward(); optimizer.step()
        scheduler.step()
        model.eval()
        all_preds, all_targets, mod_w_list = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                keyframes, motion_map, ip_vec, labels = batch['keyframes'].to(device), batch['motion_map'].to(device), batch['ip_vec'].to(device), batch['label'].to(device)
                if mode == 'ip_only': out = model(ip_vec)
                elif mode == 'rgb_only': out = model(keyframes)
                else:
                    out = model(keyframes, motion_map, ip_vec)
                    if 'modality_weights' in out: mod_w_list.append(out['modality_weights'].cpu().numpy())
                all_preds.extend(torch.argmax(out['logits'], dim=-1).cpu().numpy().tolist())
                all_targets.extend(labels.cpu().numpy().tolist())
        acc = accuracy_score(all_targets, all_preds) * 100.0
        macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0) * 100.0
        if acc >= best_acc:
            best_acc, best_f1 = acc, macro_f1
            best_preds, best_targets = np.array(all_preds), np.array(all_targets)
            if mod_w_list: best_mod_weights = np.concatenate(mod_w_list, axis=0)
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f'[{variant_name}] Epoch {epoch:02d}/{epochs:02d} | Val Acc: {acc:.2f}% | Best Acc: {best_acc:.2f}% | Macro-F1: {best_f1:.2f}%')
    if best_state is not None: model.load_state_dict(best_state)
    return {'accuracy': round(best_acc, 2), 'macro_f1': round(best_f1, 2)}, best_targets, best_preds, best_mod_weights

def save_result_figures(ablation_results, y_true, y_pred, mod_weights, output_dir):
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CRICSHOT10_CLASSES))))
    fig, ax = plt.subplots(figsize=(8.5, 7.0), dpi=200)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[c.replace('_', ' ').title() for c in CRICSHOT10_CLASSES], yticklabels=[c.replace('_', ' ').title() for c in CRICSHOT10_CLASSES], ax=ax)
    ax.set_title('CricIP-FusionNet Confusion Matrix on CricShot10 Validation Set', fontweight='bold')
    ax.set_xlabel('Predicted Cricket Shot', fontweight='bold'); ax.set_ylabel('Ground-Truth Cricket Shot', fontweight='bold')
    plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.savefig(out / 'fig_confusion_matrix.png', bbox_inches='tight'); plt.close(fig)

    names = list(ablation_results.keys())
    accs = [ablation_results[k]['accuracy'] for k in names]
    f1s = [ablation_results[k]['macro_f1'] for k in names]
    x, width = np.arange(len(names)), 0.36
    fig, ax = plt.subplots(figsize=(10, 5.2), dpi=200)
    rects1 = ax.bar(x - width/2, accs, width, label='Top-1 Accuracy (%)', color='#2b5c8f')
    ax.bar(x + width/2, f1s, width, label='Macro F1-Score (%)', color='#e07a5f')
    ax.set_ylabel('Score (%)', fontweight='bold'); ax.set_title('Ablation Study across Image Processing & Fusion Streams on CricShot10', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=18, ha='right'); ax.set_ylim(0, 108); ax.legend(); ax.grid(axis='y', alpha=0.3)
    for rect in rects1:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8.5, fontweight='bold')
    plt.tight_layout(); plt.savefig(out / 'fig_ablation_comparison.png', bbox_inches='tight'); plt.close(fig)

    if mod_weights.shape[0] == len(y_true):
        class_gates = np.zeros((len(CRICSHOT10_CLASSES), 3), dtype=np.float32)
        for c_idx in range(len(CRICSHOT10_CLASSES)):
            mask = y_true == c_idx
            class_gates[c_idx] = mod_weights[mask].mean(axis=0) if np.any(mask) else np.array([0.4, 0.35, 0.25])
        fig, ax = plt.subplots(figsize=(10, 4.8), dpi=200)
        x_cls = np.arange(len(CRICSHOT10_CLASSES))
        ax.bar(x_cls, class_gates[:, 0], label=r'$\alpha_{\mathrm{RGB}}$ (CLAHE + Crowd-Suppressed)', color='#3d5a80')
        ax.bar(x_cls, class_gates[:, 1], bottom=class_gates[:, 0], label=r'$\alpha_{\mathrm{Motion}}$ (MHI + Optical Flow)', color='#ee6c4d')
        ax.bar(x_cls, class_gates[:, 2], bottom=class_gates[:, 0]+class_gates[:, 1], label=r'$\alpha_{\mathrm{Geom}}$ (HOOF + Canny-Hough)', color='#2a9d8f')
        ax.set_xticks(x_cls); ax.set_xticklabels([c.replace('_', ' ').title() for c in CRICSHOT10_CLASSES], rotation=25, ha='right')
        ax.set_ylabel('Learned Gating Weight', fontweight='bold'); ax.set_title('Learned Gated Cross-Modal Attention Weights Across Cricket Strokes', fontweight='bold')
        ax.set_ylim(0, 1.18); ax.legend(loc='upper center', ncol=3, fontsize=8.5); plt.tight_layout(); plt.savefig(out / 'fig_modality_attention.png', bbox_inches='tight'); plt.close(fig)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset_root', type=str, default='/content/drive/MyDrive/CricShot10 dataset')
    parser.add_argument('--max_per_class', type=int, default=60)
    parser.add_argument('--epochs', type=int, default=12)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--use_demo_data', action='store_true')
    args = parser.parse_args()
    set_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using compute device:', device)
    if args.use_demo_data or not Path(args.dataset_root).exists():
        demo_root = 'data/demo_cricshot10'
        for _ in range(4): generate_demo_cricket_videos(demo_root)
        dataset_root = demo_root
    else:
        dataset_root = args.dataset_root
    samples = discover_cricshot10_videos(dataset_root, max_per_class=args.max_per_class)
    print(f'Discovered {len(samples)} total video clips across 10 classes.')
    pipeline = CricketIPPipeline()
    os.makedirs('outputs/figures', exist_ok=True); os.makedirs('checkpoints', exist_ok=True)
    save_ip_pipeline_figure(samples[0][0], samples[0][2], pipeline, 'outputs/figures/fig_ip_pipeline.png')
    labels = [s[1] for s in samples]
    train_samples, val_samples = train_test_split(samples, test_size=0.25, random_state=42, stratify=labels) if len(samples) >= 20 else (samples, samples)
    cache_dir = 'data/feature_cache'
    train_loader_raw = DataLoader(CricShot10Dataset(train_samples, pipeline, cache_dir, False, True), batch_size=args.batch_size, shuffle=True)
    val_loader_raw = DataLoader(CricShot10Dataset(val_samples, pipeline, cache_dir, False, False), batch_size=args.batch_size, shuffle=False)
    train_loader_ip = DataLoader(CricShot10Dataset(train_samples, pipeline, cache_dir, True, True), batch_size=args.batch_size, shuffle=True)
    val_loader_ip = DataLoader(CricShot10Dataset(val_samples, pipeline, cache_dir, True, False), batch_size=args.batch_size, shuffle=False)

    ablation_results = {}
    res_a, _, _, _ = train_and_evaluate_variant('A: Classical IP Only', ClassicalIPMLP(102, 128, 10), train_loader_ip, val_loader_ip, device, args.epochs, 1e-3, 'ip_only')
    ablation_results['A: Classical IP Only'] = res_a
    res_b, _, _, _ = train_and_evaluate_variant('B: Raw RGB CNN', RGBBaselineNet(10, True, False), train_loader_raw, val_loader_raw, device, args.epochs, 2e-4, 'rgb_only')
    ablation_results['B: Raw RGB CNN'] = res_b
    res_c, _, _, _ = train_and_evaluate_variant('C: +CLAHE & CrowdMask', RGBBaselineNet(10, True, True), train_loader_ip, val_loader_ip, device, args.epochs, 2e-4, 'rgb_only')
    ablation_results['C: +CLAHE & CrowdMask'] = res_c
    res_d, _, _, _ = train_and_evaluate_variant('D: +MHI & Optical Flow', CricIPFusionNet(102, 256, 10, True, False), train_loader_ip, val_loader_ip, device, args.epochs, 2e-4, 'fusion')
    ablation_results['D: +MHI & Optical Flow'] = res_d
    model_e = CricIPFusionNet(102, 256, 10, True, True)
    res_e, y_true, y_pred, mod_w = train_and_evaluate_variant('E: CricIP-FusionNet (Ours)', model_e, train_loader_ip, val_loader_ip, device, args.epochs, 2e-4, 'fusion')
    ablation_results['E: CricIP-FusionNet (Ours)'] = res_e

    torch.save({'model_state_dict': model_e.state_dict(), 'classes': CRICSHOT10_CLASSES, 'ablation_results': ablation_results}, 'checkpoints/best_cricip_fusionnet.pth')
    with open('outputs/ablation_metrics.json', 'w') as f: json.dump(ablation_results, f, indent=2)
    save_result_figures(ablation_results, y_true, y_pred, mod_w, 'outputs/figures')
    print('\n=== Final Classification Report (Proposed CricIP-FusionNet) ===')
    print(classification_report(y_true, y_pred, target_names=CRICSHOT10_CLASSES, digits=3, zero_division=0))

if __name__ == '__main__':
    main()


In [ ]:
# Step 4: Run & Visualize the 6-Stage Image Processing Pipeline on CricShot10
from src.dataset import discover_cricshot10_videos, generate_demo_cricket_videos
from src.ip_pipeline import CricketIPPipeline
from train_eval import save_ip_pipeline_figure
import matplotlib.pyplot as plt

if DATASET_ROOT is None:
    print('Generating synthetic cricket stroke clips for quick test...')
    for _ in range(4):
        generate_demo_cricket_videos('data/demo_cricshot10')
    DATASET_ROOT = 'data/demo_cricshot10'

samples = discover_cricshot10_videos(DATASET_ROOT, max_per_class=60)
print(f'Loaded {len(samples)} videos from {DATASET_ROOT}')

pipeline = CricketIPPipeline()
save_ip_pipeline_figure(samples[0][0], samples[0][2], pipeline, 'outputs/figures/fig_ip_pipeline.png')
img = plt.imread('outputs/figures/fig_ip_pipeline.png')
plt.figure(figsize=(14, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Step 5: Train All 5 Ablation Models & Generate Paper Figures
!python train_eval.py --dataset_root "{DATASET_ROOT}" --max_per_class 60 --epochs 12 --batch_size 8

In [ ]:
# Step 6: Display Generated Ablation & Confusion Matrix Figures & Launch Live Gradio UI
import gradio as gr
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.ip_pipeline import CricketIPPipeline, CRICSHOT10_CLASSES
from src.dataset import normalize_image_tensor
from src.model import CricIPFusionNet

for fig_name in ['fig_ablation_comparison.png', 'fig_confusion_matrix.png', 'fig_modality_attention.png']:
    p = Path('outputs/figures') / fig_name
    if p.exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(str(p)))
        plt.axis('off')
        plt.show()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipe_ui = CricketIPPipeline()
model_ui = CricIPFusionNet(102, 256, 10, pretrained=False).to(device)
if Path('checkpoints/best_cricip_fusionnet.pth').exists():
    ckpt = torch.load('checkpoints/best_cricip_fusionnet.pth', map_location=device)
    model_ui.load_state_dict(ckpt['model_state_dict'], strict=False)
model_ui.eval()

def gradio_predict(video_path):
    out = pipe_ui.process_video(video_path)
    kf_t = normalize_image_tensor(out['suppressed_keyframes']).unsqueeze(0).to(device)
    mot_t = normalize_image_tensor(out['motion_composite']).unsqueeze(0).to(device)
    ip_t = torch.from_numpy(out['ip_feature_vector']).unsqueeze(0).to(device)
    with torch.no_grad():
        res = model_ui(kf_t, mot_t, ip_t)
        probs = torch.softmax(res['logits'][0], dim=-1).cpu().numpy()
        mod_w = res['modality_weights'][0].cpu().numpy()
    mid = len(out['raw_keyframes']) // 2
    label_dict = {CRICSHOT10_CLASSES[i].replace('_', ' ').title(): float(probs[i]) for i in range(10)}
    gate_str = f"RGB(CLAHE): {mod_w[0]:.2f} | Motion(MHI+Flow): {mod_w[1]:.2f} | Geometry(Hough+HOOF): {mod_w[2]:.2f}"
    return (
        out['suppressed_keyframes'][mid],
        out['mhi_color'],
        out['flow_rgb'],
        out['hough_overlay'],
        label_dict,
        gate_str
    )

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Video(label='Upload Cricket Shot Video (.avi / .mp4)'),
    outputs=[
        gr.Image(label='1. LAB-CLAHE & Crowd-Suppressed ROI'),
        gr.Image(label='2. Temporal Motion History (MHI)'),
        gr.Image(label='3. Dense Farneback Optical Flow (HSV)'),
        gr.Image(label='4. Canny Edges + Hough Bat Orientation'),
        gr.Label(num_top_classes=5, label='Predicted Cricket Shot Probabilities'),
        gr.Textbox(label='Learned GCAM Modality Attention Weights')
    ],
    title='🏏 CricIP-FusionNet: EE604 Cricket Shot Recognition Demo',
    description='Upload any cricket stroke clip from CricShot10 to visualize intermediate Image Processing representations and Gated Cross-Modal Fusion predictions.'
)
demo.launch(share=True)